In [37]:
!pip install epitran --quiet
import torch
import torch.nn as nn
import torch.nn.functional as F
import epitran
import re
import os


In [38]:
# raw language files
lang_files = {
    "ar": "ar.txt",
    "fi": "fi.txt",
    "hu": "hu.txt",
    "ru": "ru.txt",
}

# Epitran converters
converters = {
    "ar": epitran.Epitran("ara-Arab"),
    "fi": epitran.Epitran("fin-Latn"),
    "hu": epitran.Epitran("hun-Latn"),
    "ru": epitran.Epitran("rus-Cyrl"),
}

def clean_input_text(text):
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\d+", "", text)
    return text.strip()


def convert_to_ipa(file_name, converter):
    ipa_words = []

    with open(file_name, "r", encoding="utf-8") as f:
        for line in f:
            word = clean_input_text(line)

            if not word:
                continue

            try:
                ipa = converter.transliterate(word).strip()

                if ipa:
                    ipa_words.append(ipa)

            except Exception as e:
                print(f"failed: {word} -> {e}")

    return ipa_words


ipa_by_lang = {}

for lang, file_name in lang_files.items():
    ipa_by_lang[lang] = convert_to_ipa(
        file_name,
        converters[lang]
    )

print(ipa_by_lang)



{'ar': ['،', 'laː', 'mn', 'fiː', 'aٔn', 'hðaː', 'ʕlى', 'maː', 'aٔnaː', 'hl', 'uː', 'iːaː', 'ðlk', 'lqd', 'lm', 'maːðaː', 'kaːn', 'hnaː', 'aٕlى', 'aٔnt', 'huː', 'hðh', 'ʕn', 'nʕm', 'an', 'hnaːk', 'kl', 'ħsnaːً', 'liːs', 'knt', 'fqtˤ', 'ʃiːء', 'alʔaːn', 'mʕ', 'alðiː', 'lkn', 'aٔd͡ʒl', 'lk', 'iːd͡ʒb', 'ln', 'kiːf', 'ħsnaː', 'anaː', 'liː', 'aٕnh', 'suːf', 'nħn', 'aٕðaː', 'kaːnt', 'ʕndmaː', 'aٔiː', 'qd', 'lmaːðaː', 'hiː', 'alʔamr', 'aٔnh', 'fى', 'ħtى', 'bʕd', 'kðlk', 'aٔuː', 'uːlkn', 'qbl', 'altiː', 'anh', 'aٕnhaː', 'ـ', 'luː', 'rbmaː', 'aٔiːn', 'hiːaː', 'tlk', 'aٔʕrf', 'alى', 'aٕn', 'iːkuːn', 'bʕdˤ', 'aٔriːd', 'aٔʕtqd', 'ant', 'ʕliːk', 'mθl', 'sˤħiːħ', 'bh', 'aٔnk', 'aluːqt', 'bxiːr', 'aٔʕlm', 'aٔħd', 'ʃxsˤ', 'd͡ʒdaːً', 'rd͡ʒl', 'ʃkraːً', 'lðaː', 'iːmkn', 'kmaː', 'klaː', 'aٔkθr', 'iːbduː', 'd͡ʒiːd', 'tkuːn', 'ʕliː', 'aliːuːm', 'iːmknk', 'anhaː', 'aٔliːs', 'mrħbaːً', 'ldiː', 'ʕliːh', 'aٓxr', 'lst', 'mnð', 'ɣiːr', 'aٔiːhaː', 'ldiːk', 'ħqaːً', 'aٔntِ', 'triːd', 'aٔsttˤiːʕ', 'alrd͡ʒl', 'tʕrf',

In [39]:
# Tokenizer
special_tokens = ["<PAD>", "<BOS>", "<EOS>"]
chars = []
for lang, ipa_list in ipa_by_lang.items():
  for ipa in ipa_list:
    chars.extend(ipa)

ipa_list = sorted(list(set(chars)))
print(len(ipa_list))
print(ipa_list)

tokens = special_tokens + ipa_list

id2ipa = {i: token for i, token in enumerate(tokens)}
ipa2id = {token: i for i, token in enumerate(tokens)}

PAD_ID = ipa2id["<PAD>"]
BOS_ID = ipa2id["<BOS>"]
EOS_ID = ipa2id["<EOS>"]

print("vocab:", len(tokens))
print(ipa2id)

79
['-', '.', ':', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'æ', 'ç', 'ð', 'ø', 'ħ', 'ŋ', 'ƒ', 'ɑ', 'ɒ', 'ɕ', 'ɛ', 'ɟ', 'ɡ', 'ɣ', 'ɦ', 'ɨ', 'ɲ', 'ʂ', 'ʃ', 'ʋ', 'ʒ', 'ʔ', 'ʕ', 'ʲ', 'ː', 'ˤ', '̪', '͡', 'θ', 'ћ', '،', '؛', 'ء', 'ة', 'ـ', 'ى', 'ً', 'ٌ', 'ٍ', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', 'ٓ', 'ٔ', 'ٕ', 'ٹ', 'ھ', 'ḱ']
vocab: 82
{'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, '-': 3, '.': 4, ':': 5, 'a': 6, 'b': 7, 'c': 8, 'd': 9, 'e': 10, 'f': 11, 'g': 12, 'h': 13, 'i': 14, 'j': 15, 'k': 16, 'l': 17, 'm': 18, 'n': 19, 'o': 20, 'p': 21, 'q': 22, 'r': 23, 's': 24, 't': 25, 'u': 26, 'v': 27, 'w': 28, 'x': 29, 'y': 30, 'z': 31, 'æ': 32, 'ç': 33, 'ð': 34, 'ø': 35, 'ħ': 36, 'ŋ': 37, 'ƒ': 38, 'ɑ': 39, 'ɒ': 40, 'ɕ': 41, 'ɛ': 42, 'ɟ': 43, 'ɡ': 44, 'ɣ': 45, 'ɦ': 46, 'ɨ': 47, 'ɲ': 48, 'ʂ': 49, 'ʃ': 50, 'ʋ': 51, 'ʒ': 52, 'ʔ': 53, 'ʕ': 54, 'ʲ': 55, 'ː': 56, 'ˤ': 57, '̪': 58, '͡': 59, 'θ': 60, 'ћ': 61, '،': 62, '؛': 63, 'ء': 64, 'ة'

In [40]:
lang2id = {
    "ar": 0,
    "fi": 1,
    "hu": 2,
    "ru": 3,
}
block_size = 24
def encode_word(word):
    ids = [BOS_ID]

    for char in word:
        ids.append(ipa2id[char])

    ids.append(EOS_ID)

    # 最大長を超えたら切る
    ids = ids[:block_size + 1]

    # PADで長さをそろえる
    ids += [PAD_ID] * (block_size + 1 - len(ids))

    # 次文字予測なので1個ずらす
    x = torch.tensor(ids[:-1], dtype=torch.long)
    y = torch.tensor(ids[1:], dtype=torch.long)

    return x, y
samples = []

for lang, words in ipa_by_lang.items():
    lang_id = lang2id[lang]

    for word in words:
        x, y = encode_word(word)

        samples.append(
            (x, y, lang_id)
        )

In [41]:
import random

batch_size = 32

def get_batch():
    batch = random.sample(samples, batch_size)

    x = torch.stack([
        sample[0]
        for sample in batch
    ])

    y = torch.stack([
        sample[1]
        for sample in batch
    ])

    lang_ids = torch.tensor([
        sample[2]
        for sample in batch
    ], dtype=torch.long)

    return x, y, lang_ids
xb, yb, lang_ids = get_batch()

print(xb.shape)
print(yb.shape)
print(lang_ids.shape)
print(lang_ids[:10])

torch.Size([32, 24])
torch.Size([32, 24])
torch.Size([32])
tensor([1, 2, 1, 1, 0, 0, 0, 1, 0, 3])


In [42]:
n_embd = 64
vocab_size = len(ipa2id)
num_langs = len(lang2id)

class CharTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # NEW
        self.language_embedding = nn.Embedding(
            num_langs,
            n_embd
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=4,
            dim_feedforward=128,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, lang_ids=None, targets=None, lang_weights=None):
        B, T = idx.shape

        # IPA char embedding
        tok_emb = self.token_embedding_table(idx)

        # position embedding
        pos = torch.arange(
            T,
            dtype=torch.long,
            device=idx.device
        )

        pos_emb = self.position_embedding_table(pos)

        # language embedding
        if lang_weights is None:
              lang_emb = self.language_embedding(lang_ids)

        else:
            lang_emb = (
                lang_weights
                @ self.language_embedding.weight
            )

        # [B, n_embd]
        # ↓
        # [B, 1, n_embd]
        lang_emb = lang_emb[:, None, :]

        # combine
        x = tok_emb + pos_emb + lang_emb

        mask = nn.Transformer.generate_square_subsequent_mask(
            T,
            device=idx.device
        )

        x = self.transformer(
            x,
            mask=mask,
            is_causal=True
        )

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits_flat = logits.reshape(B * T, C)
            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(
                logits_flat,
                targets_flat,
                ignore_index=PAD_ID
            )

        return logits, loss
model = CharTransformer()

xb, yb, lang_ids = get_batch()

logits, loss = model(
    xb,
    lang_ids,
    yb
)

print(logits.shape)
print(loss)

torch.Size([32, 24, 82])
tensor(4.4775, grad_fn=<NllLossBackward0>)


In [43]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

max_iters = 3000
eval_interval = 300

for iter in range(max_iters):

    xb, yb, lang_ids = get_batch()

    logits, loss = model(xb,lang_ids, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    optimizer.step()

    if iter % eval_interval == 0:
        print(f"Step {iter}: Loss = {loss.item():.4f}")

print(f"🎉 学習完了！最終Loss: {loss.item():.4f}")

Step 0: Loss = 4.4586
Step 300: Loss = 2.3288
Step 600: Loss = 2.3580
Step 900: Loss = 2.2598
Step 1200: Loss = 2.1422
Step 1500: Loss = 2.0586
Step 1800: Loss = 2.2115
Step 2100: Loss = 2.0987
Step 2400: Loss = 2.1649
Step 2700: Loss = 2.2328
🎉 学習完了！最終Loss: 2.1020


In [45]:
@torch.no_grad()
def generate_word(lang, temperature=0.8, max_len=20):
    model.eval()

    # 例: "ar" -> 0
    lang_id = torch.tensor(
        [lang2id[lang]],
        dtype=torch.long
    )

    # <BOS> から開始
    context = torch.tensor(
        [[BOS_ID]],
        dtype=torch.long
    )

    generated = []

    for _ in range(max_len):

        # block_sizeを超えたら後ろだけ使う
        idx_cond = context[:, -block_size:]

        logits, _ = model(
            idx_cond,
            lang_id
        )

        # 最後の位置だけ見る
        logits = logits[:, -1, :] / temperature

        # PADとBOSは生成させない
        logits[:, PAD_ID] = float("-inf")
        logits[:, BOS_ID] = float("-inf")

        probs = F.softmax(logits, dim=-1)

        next_id = torch.multinomial(
            probs,
            num_samples=1
        )

        token_id = next_id.item()

        # finish when EOS appeared
        if token_id == EOS_ID:
            break

        generated.append(id2ipa[token_id])

        context = torch.cat(
            (context, next_id),
            dim=1
        )

    return "".join(generated)
print(generate_word("ar"))
print(generate_word("fi"))
print(generate_word("hu"))
print(generate_word("ru"))

muːn
jopɑ:
mɒɡɒdɒkon
muma


In [46]:
# mixing libraries
def make_lang_weights(mix):

    weights = torch.zeros(
        1,
        len(lang2id)
    )

    for lang, ratio in mix.items():
        weights[0, lang2id[lang]] = ratio

    weights = weights / weights.sum()

    return weights
weights = make_lang_weights({
    "ar": 0.7,
    "fi": 0.3
})

In [48]:
@torch.no_grad()
def generate_mixed(mix, temperature=0.8, max_len=20):

    model.eval()

    lang_weights = make_lang_weights(mix)

    context = torch.tensor(
        [[BOS_ID]],
        dtype=torch.long
    )

    generated = []

    for _ in range(max_len):

        idx_cond = context[:, -block_size:]

        logits, _ = model(
            idx_cond,
            lang_weights=lang_weights
        )

        logits = logits[:, -1, :] / temperature

        logits[:, PAD_ID] = float("-inf")
        logits[:, BOS_ID] = float("-inf")

        probs = F.softmax(logits, dim=-1)

        next_id = torch.multinomial(
            probs,
            num_samples=1
        )

        token_id = next_id.item()

        if token_id == EOS_ID:
            break

        generated.append(id2ipa[token_id])

        context = torch.cat(
            (context, next_id),
            dim=1
        )

    return "".join(generated)
generate_mixed({
    "ar": 0.5,
    "fi": 0.25,
    "ru": 0.25,
})

'almyd'

In [50]:
for _ in range(30):
    print(generate_mixed({
        "hu": 0.5,
        "fi": 0.25,
        "ru": 0.25,
    }))

kieːaːtɒ
mdoɡik
veːrni
porbolorto
poloɡin
jut͡ɕonɒ
lupdol
oti
koin
meːstɛ
bolʲin
t͡sinaːlt
tut͡sim
meːɡen
tut͡sik
tut͡seː
suɡlaːd
hut͡ɕ
sobondo
tozal
raːlaːt
leːvɛ
teːsɛt
volojt͡s
sobut
fomomb
mukan
seːrɡ
mreɡos
kento͡selm
